# ECG Digitzation

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))
        

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from sklearn.model_selection import train_test_split
import torch
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import cv2
import copy
import easyocr
import random
from scipy.ndimage import gaussian_filter
from scipy.signal import resample
from scipy.interpolate import interp1d

In [ ]:
# Reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = seed + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [ ]:
path = '/kaggle/input/physionet-ecg-image-digitization/train'
metadata_df = pd.read_csv('/kaggle/input/physionet-ecg-image-digitization/train.csv')

# Split data for mask segmentation and grid removal tasks
train_grid, train_mask = train_test_split(metadata_df, test_size=0.1, random_state=42)

# Split data into train and val
train_grid, val_grid = train_test_split(train_grid, test_size=0.2, random_state=42)
train_mask, val_mask = train_test_split(train_mask, test_size=0.2, random_state=42)

# Evaluation suffixes
val_suffix_pool = [3, 4, 5, 6, 9, 10, 11, 12]
val_suffix_map = {}

rng = np.random.default_rng(seed=0)   

for id_str in val_grid["id"].astype(str):
    val_suffix_map[id_str] = int(rng.choice(val_suffix_pool))

# Global Functions

### Resizing

In [ ]:
def resize_with_padding(img, target_h=1700, target_w=2200):
        h, w = img.shape[:2]
        scale = min(target_w / w, target_h / h)
    
        new_w = int(w * scale)
        new_h = int(h * scale)
    
        # Resize while keeping aspect ratio
        resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
        # If grayscale or BGR
        if img.ndim == 2:
            padded = np.zeros((target_h, target_w), dtype=img.dtype)
        else:
            padded = np.zeros((target_h, target_w, img.shape[2]), dtype=img.dtype)
    
        pad_x = (target_w - new_w) // 2
        pad_y = (target_h - new_h) // 2
    
        padded[pad_y:pad_y+new_h, pad_x:pad_x+new_w] = resized
    
        return padded

def downsample(img):
    target_h = 512
    raw_w = int(target_h * (2200/1700))
    target_w = int(np.round(raw_w / 16) * 16)   # → 656
    
    image_small = cv2.resize(img, (target_w, target_h),interpolation=cv2.INTER_AREA)  

    return image_small

In [ ]:
def order_corners_clockwise(pts):

    pts = np.array(pts, dtype=np.float32)

    # sort by y first, then x
    y_sorted = pts[np.argsort(pts[:, 1])]

    # top two points
    top = y_sorted[:2]
    bottom = y_sorted[2:]

    # sort top by x (TL, TR)
    tl, tr = top[np.argsort(top[:, 0])]

    # sort bottom by x (BL, BR)
    bl, br = bottom[np.argsort(bottom[:, 0])]

    return np.array([tl, tr, br, bl], dtype=np.float32)



### UNet

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module): 
    def __init__(self, chs=[64, 128, 256, 512, 1024]): 
        super().__init__() 
        self.down1 = DoubleConv(1, chs[0]) 
        self.down2 = DoubleConv(chs[0], chs[1]) 
        self.down3 = DoubleConv(chs[1], chs[2]) 
        self.down4 = DoubleConv(chs[2], chs[3]) 
        self.down5 = DoubleConv(chs[3], chs[4]) 
        
        self.pool = nn.MaxPool2d(2) 
        
        self.up4 = nn.ConvTranspose2d(chs[4], chs[3], 2, stride=2) 
        self.conv4 = DoubleConv(chs[4], chs[3]) 
        
        self.up3 = nn.ConvTranspose2d(chs[3], chs[2], 2, stride=2) 
        self.conv3 = DoubleConv(chs[3], chs[2]) 
        
        self.up2 = nn.ConvTranspose2d(chs[2], chs[1], 2, stride=2) 
        self.conv2 = DoubleConv(chs[2], chs[1]) 
        
        self.up1 = nn.ConvTranspose2d(chs[1], chs[0], 2, stride=2) 
        self.conv1 = DoubleConv(chs[1], chs[0]) 
        self.out = nn.Conv2d(chs[0], 1, 1) 
        
    def forward(self, x): 
        # Encoder 
        d1 = self.down1(x) # 1 -> 64 
        p1 = self.pool(d1) 
        d2 = self.down2(p1) # 64 -> 128 
        p2 = self.pool(d2) 
        d3 = self.down3(p2) # 128 -> 256 
        p3 = self.pool(d3)
        d4 = self.down4(p3) # 256 -> 512 
        p4 = self.pool(d4)
        bottleneck = self.down5(p4) # 512 -> 1024 
        
        # Decoder 
        u4 = self.up4(bottleneck) 
        u4 = torch.cat([u4, d4], dim=1) 
        x = self.conv4(u4) 

        u3 = self.up3(x) 
        u3 = torch.cat([u3, d3], dim=1) 
        x = self.conv3(u3) 

        u2 = self.up2(x) 
        u2 = torch.cat([u2, d2], dim=1) 
        x = self.conv2(u2) 
        
        u1 = self.up1(x) 
        u1 = torch.cat([u1, d1], dim=1) 
        x = self.conv1(u1) 
        return self.out(x)

class DiceLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        pred = torch.sigmoid(pred)      
        num = 2 * (pred * target).sum(dim=(2,3))
        den = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
        dice = 1 - (num + self.eps) / (den + self.eps)
        return dice.mean()
        


### Attention ResUNet

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)

        self.residual = (
            nn.Conv2d(in_ch, out_ch, 1)
            if in_ch != out_ch else nn.Identity()
        )

    def forward(self, x):
        res = self.residual(x)
        x = self.relu(self.conv1(x))
        x = self.conv2(x)
        return self.relu(x + res)

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        
        # Channel Attention 
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )

        # Spatial Attention
        self.spatial = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False),
            nn.Sigmoid()
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        # Channel Attention 
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        channel_att = self.sigmoid(avg_out + max_out)
        x = x * channel_att

        # Spatial Attention 
        avg_ch = torch.mean(x, dim=1, keepdim=True)
        max_ch, _ = torch.max(x, dim=1, keepdim=True)
        spatial_att = self.spatial(torch.cat([avg_ch, max_ch], dim=1))

        return x * spatial_att


class UNet_Res_CBAM(nn.Module):
    def __init__(self, chs=[64, 128, 256, 512, 1024]):
        super().__init__()

        # Encoder
        self.down1 = ResBlock(2, chs[0])
        self.down2 = ResBlock(chs[0], chs[1])
        self.down3 = ResBlock(chs[1], chs[2])
        self.down4 = ResBlock(chs[2], chs[3])
        self.down5 = ResBlock(chs[3], chs[4])

        self.pool = nn.MaxPool2d(2)

        # CBAM applied to skip connections
        self.cbam4 = CBAM(chs[3])
        self.cbam3 = CBAM(chs[2])
        self.cbam2 = CBAM(chs[1])
        self.cbam1 = CBAM(chs[0])

        # Decoder
        self.up4 = nn.ConvTranspose2d(chs[4], chs[3], 2, stride=2)
        self.conv4 = ResBlock(chs[4], chs[3])

        self.up3 = nn.ConvTranspose2d(chs[3], chs[2], 2, stride=2)
        self.conv3 = ResBlock(chs[3], chs[2])

        self.up2 = nn.ConvTranspose2d(chs[2], chs[1], 2, stride=2)
        self.conv2 = ResBlock(chs[2], chs[1])

        self.up1 = nn.ConvTranspose2d(chs[1], chs[0], 2, stride=2)
        self.conv1 = ResBlock(chs[1], chs[0])

        self.out = nn.Conv2d(chs[0], 1, 1)

    def forward(self, x):

        # Encoder
        d1 = self.down1(x)
        p1 = self.pool(d1)

        d2 = self.down2(p1)
        p2 = self.pool(d2)

        d3 = self.down3(p2)
        p3 = self.pool(d3)

        d4 = self.down4(p3)
        p4 = self.pool(d4)

        bottleneck = self.down5(p4)

        # Decoder

        u4 = self.up4(bottleneck)
        d4_att = self.cbam4(d4)         
        u4 = torch.cat([u4, d4_att], 1)
        x = self.conv4(u4)

        u3 = self.up3(x)
        d3_att = self.cbam3(d3)
        u3 = torch.cat([u3, d3_att], 1)
        x = self.conv3(u3)

        u2 = self.up2(x)
        d2_att = self.cbam2(d2)
        u2 = torch.cat([u2, d2_att], 1)
        x = self.conv2(u2)

        u1 = self.up1(x)
        d1_att = self.cbam1(d1)
        u1 = torch.cat([u1, d1_att], 1)
        x = self.conv1(u1)

        return self.out(x)


class DiceFocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0, dice_weight=0.5, focal_weight=0.5, smooth=1e-6):

        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.dice_weight = dice_weight
        self.focal_weight = focal_weight
        self.smooth = smooth

    def dice_loss(self, pred, target):
        pred = torch.sigmoid(pred)

        intersection = (pred * target).sum(dim=(2, 3))
        union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))

        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        dice_loss = 1 - dice.mean()

        return dice_loss

    def focal_loss(self, pred, target):
        bce = F.binary_cross_entropy_with_logits(pred, target, reduction="none")
        focal = bce.mean()

        return focal

    def forward(self, pred, target):
        dice = self.dice_loss(pred, target)
        focal = self.focal_loss(pred, target)

        loss = self.dice_weight * dice + self.focal_weight * focal
        return loss

# ORB Alignment

In [ ]:
class ORB:
    def __init__(self,
                 match_ratio=0.7,
                 ransac_thresh=5.0,
                 full_shape = (1700,2200),
                 n_features=1000):

        self.match_ratio = match_ratio
        self.ransac_thresh = ransac_thresh
        self.orb = cv2.ORB_create(nfeatures=n_features)
        self.bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
        self.full_shape = full_shape
    
    def keypoint_descriptors(self, image):
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        kp, des = self.orb.detectAndCompute(gray, None)
        return kp, des

    def call_kp(self, kp1, des1, lq_image):
        lq_image = resize_with_padding(lq_image)
        kp2, des2 = self.keypoint_descriptors(lq_image)
        matches = self.bf.knnMatch(des1, des2, k=2)

        good_matches = []
        for m, n in matches:
            if m.distance < self.match_ratio * n.distance:
                good_matches.append(m)

        if len(good_matches) < 4:
            print("Not enough matches found!")
            return None, None

        src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
        dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

        H, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, self.ransac_thresh)

        h, w = self.full_shape
        
        if (H is None or H.shape != (3, 3)):
            return None, None
            
        else:
            aligned = cv2.warpPerspective(lq_image, H, (w, h))
            return aligned, H

    def __call__(self, hq_image, lq_image):
        kp1, des1 = self.keypoint_descriptors(hq_image)
        return self.call_kp(kp1, des1, lq_image)


In [ ]:
suffixes = [3, 4, 5, 6, 9, 10, 11, 12]
align = ORB() 

def precompute_H(metadata):
    records = []
    for _, row in tqdm(metadata.iterrows(), total=len(metadata)):
        id_str = str(row["id"])
        hq_path = f"{path}/{id_str}/{id_str}-0001.png"
        hq_image = cv2.imread(hq_path)
        kp1, ds1 = align.keypoint_descriptors(hq_image)
    
        for suffix in suffixes:
            lq_path = f"{path}/{id_str}/{id_str}-{suffix:04d}.png"
            lq_image = cv2.imread(lq_path)

            _, H = align.call_kp(kp1, ds1, lq_image)
            
            if H is not None:
                w=2200
                h=1700
                src = np.float32([[0,0], [w-1,0], [w-1,h-1], [0,h-1]])
                H_inv = np.linalg.inv(H)

                dst_corners = cv2.perspectiveTransform(src.reshape(-1,1,2), H_inv)
                dst_corners = dst_corners.reshape(4, 2).astype(np.float32)

                record = {"id": id_str, "suffix": suffix}

                # store corner coordinates
                for i, (x, y) in enumerate(dst_corners):
                    record[f"x{i}"] = float(x)
                    record[f"y{i}"] = float(y)

                pts = np.array([
                    [row.x0, row.y0],
                    [row.x1, row.y1],
                    [row.x2, row.y2],
                    [row.x3, row.y3],
                ], dtype=np.float32)

                records.append(record)
    
    return pd.DataFrame(records)

H_train = precompute_H(train_mask)
H_train.to_csv('/kaggle/working/H_train.csv', index=False)
H_val = precompute_H(val_mask)
H_val.to_csv('/kaggle/working/H_val.csv', index=False)


In [ ]:
# UNet 1 Mean + Std
'''

means = []
stds  = []

for id_str in train_mask['id']:
    folder = os.path.join(path, str(id_str))
    for fname in os.listdir(folder):
        if not fname.endswith(".png"):
            continue
        
        img = cv2.imread(os.path.join(folder, fname), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        means.append(img.mean())
        stds.append(img.std())

global_mean = np.mean(means)
global_std  = np.sqrt(np.mean(np.square(stds)))

print(global_mean, global_std)

'''

In [ ]:
UNET1_MEAN = 193
UNET1_STD = 55

In [ ]:
class CropDataset(Dataset):
    def __init__(self,
                 metadata: pd.DataFrame,
                 H: pd.DataFrame,
                 path: str,
                 suffix_map=None):

        self.metadata = metadata.copy().reset_index(drop=True)
        self.H = H
        self.path = path

        self.image_suffixes = (3, 4, 5, 6, 9, 10, 11, 12)

        self.samples = []
        for _, row in self.metadata.iterrows():
            id_str = str(row["id"])
            for suffix in self.image_suffixes:
                self.samples.append({
                    "id": id_str,
                    "suffix": suffix
                })
            

    def __len__(self):
        return len(self.samples)

    def load_image(self, path): 
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        return img

    def to_tensor(self, img):
        tensor = transforms.ToTensor()(img)
        tensor = transforms.Normalize(mean=(UNET1_MEAN / 255.0), std=(UNET1_STD / 255.0))(tensor)
        return tensor

    def __getitem__(self, idx):

        sample_info = self.samples[idx]
        id_str = sample_info['id']
        suffix = sample_info['suffix']
    
        img_path = f"{self.path}/{id_str}/{id_str}-{suffix:04d}.png"
        lq_image = self.load_image(img_path)
        lq_image = resize_with_padding(lq_image)                     
        lq_image_small = downsample(lq_image)         
        lq_tensor = self.to_tensor(lq_image_small)
        
        row = self.H[(self.H.id == id_str) & (self.H.suffix == suffix)].iloc[0]
        pts = np.array([
            [row.x0, row.y0],
            [row.x1, row.y1],
            [row.x2, row.y2],
            [row.x3, row.y3],
        ], dtype=np.float32)
        
        mask = np.zeros((1700, 2200), dtype=np.uint8)
        cv2.fillPoly(mask, [pts.astype(np.int32)], 1)
        
        mask_small = downsample(mask)
        mask_tensor = torch.from_numpy(mask_small).unsqueeze(0).float()

        return {
            "id": id_str,
            "suffix": suffix,
            "img": lq_tensor,
            "mask": mask_tensor
        }


In [ ]:
# Dataset and Dataloader
train_dataset = CropDataset(metadata=train_mask, path=path)#, H=H_train)
val_dataset = CropDataset(metadata=val_mask, path=path)#, H=H_val)

high_weight_suffixes = {5, 6, 9, 10}

weights = []
for s in train_dataset.samples:
    if s["suffix"] in high_weight_suffixes:
        weights.append(3.0)  # upweight
    else:
        weights.append(1.0)  # normal weight

from torch.utils.data import WeightedRandomSampler

g = torch.Generator().manual_seed(42)

sampler = WeightedRandomSampler(
    weights=weights,
    num_samples=len(weights), 
    replacement=True,
    generator=g
)

train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler, worker_init_fn=seed_worker)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, worker_init_fn=seed_worker)

In [ ]:
torch.cuda.empty_cache()
torch.cuda.synchronize()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = UNet().to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-3)

#optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9) #original study with longer epochs
criterion_bce = nn.BCEWithLogitsLoss()
criterion_dice = DiceLoss()

def combined_loss(pred, target):
    return 0.5*criterion_bce(pred, target) + 0.5*criterion_dice(pred, target)

num_epochs = 10
best_val_loss = float("inf")
train_losses, val_losses = [], []
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(num_epochs):
    # TRAINING
    model.train()
    total_train_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for batch in pbar:
        imgs = batch["img"].to(device)        # (B,3,H,W)
        masks = batch["mask"].to(device)      # (B,1,H,W)

        optimizer.zero_grad()
        pred = model(imgs)                    # logits output

        loss = combined_loss(pred, masks)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # VALIDATION
    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
        for batch in pbar:
            imgs = batch["img"].to(device)
            masks = batch["mask"].to(device)

            pred = model(imgs)
            loss = combined_loss(pred, masks)

            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}")

    # SAVE BEST MODEL
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_wts = copy.deepcopy(model.state_dict())


# load best weights
model.load_state_dict(best_model_wts)
model.eval()

plt.figure(figsize=(8,5))
plt.plot(train_losses, label="Train Loss", marker='o')
plt.plot(val_losses, label="Val Loss", marker='s')
plt.title("UNet Training Curve (Dice + BCE Loss)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()



In [ ]:
# Visualise predictions

def denorm(img_tensor):
    img = img_tensor.cpu().numpy()
    img = img * UNET1_STD/255 + UNET1_MEAN/255        
    img = np.clip(img, 0, 1)
    img = (img * 255).astype(np.uint8)
    img = img.transpose(1,2,0)  
    if img.shape[-1] == 1:
        img = np.repeat(img, 3, axis=-1)
    return img

val_iter = iter(val_loader)
batch = next(val_iter)
imgs       = batch["img"].to(device)     # (B,1,H,W)
true_masks = batch["mask"].to(device)    # (B,1,H,W)

model.eval()
with torch.no_grad():
    logits    = model(imgs)
    preds     = torch.sigmoid(logits)
    preds_bin = (preds > 0.5).float()

N = min(6, imgs.shape[0]) 

plt.figure(figsize=(16, 4 * N))

for i in range(N):
    img_np = denorm(imgs[i])

    mask_pred = preds_bin[i, 0].cpu().numpy().squeeze()
    mask_true = true_masks[i, 0].cpu().numpy().squeeze()

    overlay_pred = img_np.copy()
    overlay_pred[mask_pred == 1] = [255, 0, 0]

    overlay_true = img_np.copy()
    overlay_true[mask_true == 1] = [0, 255, 0]

    row = i

    plt.subplot(N, 3, row*3+1)
    plt.imshow(img_np)
    plt.title(f"Original #{i}")
    plt.axis("off")

    plt.subplot(N, 3, row*3+2)
    plt.imshow(overlay_pred)
    plt.title(f"Pred Mask #{i} (Red)")
    plt.axis("off")

    plt.subplot(N, 3, row*3+3)
    plt.imshow(overlay_true)
    plt.title(f"GT Mask #{i} (Green)")
    plt.axis("off")

plt.tight_layout()
plt.show()



In [ ]:
torch.save(model.state_dict(), "/kaggle/working/unet1_best.pth")

In [ ]:
unet1_path = "/kaggle/input/unet1-crop-doc/unet1_best.pth"
model = UNet()  
model.load_state_dict(torch.load(unet1_path, map_location="cpu"))
model.eval()

In [ ]:
!pip install -q gdown
import gdown

file_id = "1qaJeb7rBfyxsERlRaj9Md2aspaZKs-Tt"  # lr = 0.001
url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(url, "model.pth", quiet=False)

model = UNet()  
model.load_state_dict(torch.load("model.pth", map_location="cpu"))
model.eval()

# Grid Removal

In [ ]:
# UNet 2 Mean + Std

'''
gray_means = []
gray_stds = []
fft_means = []
fft_stds = []

suffixes = (3, 4, 5, 6, 9, 10, 11, 12)
total_iters = len(train['id']) * len(suffixes)
pbar = tqdm(total=total_iters)

count = 0
for id_str in train_grid['id']:
    for suffix in suffixes:
        img_path = f"{path}/{id_str}/{id_str}-{suffix:04d}.png"
        hq_path  = f"{path}/{id_str}/{id_str}-0001.png"
    
        img = cv2.imread(img_path)
        hq_img = cv2.imread(hq_path)

        padded = resize_with_padding(img)
        aligned, Hk = ORB()(hq_img, padded)  

        # grayscale stats
        gray_means.append(aligned.mean())
        gray_stds.append(aligned.std())

        # FFT
        f = np.fft.fftshift(np.fft.fft2(aligned))
        mag = np.log(np.abs(f)+1)
        fft_means.append(mag.mean())
        fft_stds.append(mag.std())

        count += 1
        pbar.update(1)

        if count % 100 == 0:
            mean_gray = np.mean(gray_means)
            std_gray  = np.sqrt(np.mean(np.square(gray_stds)))

            mean_fft = np.mean(fft_means)
            std_fft  = np.sqrt(np.mean(np.square(fft_stds)))

            pbar.write(
                f"[{count} images] \nGray: mean={mean_gray:.4f}, std={std_gray:.4f} | \nFFT: mean={mean_fft:.4f}, std={std_fft:.4f}")

pbar.close()

mean_gray = np.mean(gray_means)
std_gray  = np.sqrt(np.mean(np.square(gray_stds)))

mean_fft = np.mean(fft_means)
std_fft  = np.sqrt(np.mean(np.square(fft_stds)))

print("Gray:", mean_gray, std_gray)
print("FFT :", mean_fft, std_fft)

'''

In [ ]:
UNET2_MEAN = 196/255
UNET2_STD = 42/255

FFT_MEAN = 4.58
FFT_STD = 2.36

In [ ]:
def render_ecg(
    signals,                
    fs,
    row_y=[580, 862, 1150, 1405],                  
    col_x=[125, 618, 1108, 1600],                  
    image_h=1700,
    image_w=2200,
    seconds_per_panel=2.5,
    px_per_mv=78,           # 38 px = 0.5 mV
    px_per_sec=196,          # 38 px = 0.2 s
    key=0
):

    # Ensure signals is NumPy array
    signals = np.asarray(signals, dtype=float)
    fs = float(fs)

    # Black background
    img = np.zeros((image_h, image_w), dtype=np.uint8)

    # Lead layout
    layout = {
        0: ["I",   "aVR", "V1", "V4"],
        1: ["II",  "aVL", "V2", "V5"],
        2: ["III", "aVF", "V3", "V6"],
        3: ["II_FULL"]
    }
    

    lead_idx = {
        "I":0, "II":1, "III":2,
        "aVR":3, "aVL":4, "aVF":5,
        "V1":6, "V2":7, "V3":8, "V4":9, "V5":10, "V6":11
    }

    n_panel = int(seconds_per_panel * fs)
    n_full = signals.shape[0]
    panel_h = int((row_y[1] - row_y[0]) * 0.8)
    panel_h = 250

    def extract_lead(signal, n_required):
        valid_mask = ~np.isnan(signal)
        if valid_mask.sum() < 5:
            return np.zeros(n_required)
        valid_sig = signal[valid_mask]
        if len(valid_sig) >= n_required:
            return valid_sig[:n_required]
        return np.concatenate([valid_sig, np.zeros(n_required - len(valid_sig))])

    def draw_trace(signal, start_x, start_y):
        n_samples = len(signal)
        t = np.arange(n_samples) / fs
        x = np.round(t * px_per_sec).astype(int) + start_x
        baseline = start_y + panel_h // 2
        y = (baseline - signal * px_per_mv).astype(int)
        valid = (x >= 0) & (x < image_w)
        x = x[valid]
        y = y[valid]
        for i in range(len(x)-1):
            cv2.line(img, (x[i], y[i]), (x[i+1], y[i+1]), color=255, thickness=3)

    # Draw all panels
    traces = layout[key]
    if key < 3:
        for c, lead_name in enumerate(traces):
            yi = row_y[key]
            xi = col_x[c]
            idx = lead_idx[lead_name]
            trace = extract_lead(signals[:, idx], n_panel)
            draw_trace(trace, xi, yi)
    else:
        # Full 10s lead II
        idx = lead_idx["II"]
        trace = extract_lead(signals[:, idx], n_full)
        yi = row_y[3]
        xi = col_x[0]
        draw_trace(trace, xi, yi)

    return img

class ECGDataset(Dataset):
    def __init__(self, metadata: pd.DataFrame, path: str, suffix_map=None):
        self.metadata = metadata.reset_index(drop=True)
        self.path = path
        self.ids = self.metadata["id"].tolist()
        self.fs = self.metadata["fs"].tolist()
        self.suffix_map = suffix_map
        self.suffixes = (3, 4, 5, 6, 9, 10, 11, 12)

        # Oversample stained images
        self.suffixes = (3, 4, 5, 6, 9, 9, 9, 10, 10, 10, 11, 11, 11, 12, 12, 12)
        
    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        id_str = str(self.ids[idx])
        fs = self.fs[idx]
        suffix = self.suffix_map[id_str] if self.suffix_map else np.random.choice(self.suffixes)

        lq_img = cv2.imread(f"{self.path}/{id_str}/{id_str}-{suffix:04d}.png", cv2.IMREAD_GRAYSCALE)
        lq_img = resize_with_padding(lq_img)

        return {
            "id": id_str,
            "fs": fs,
            "lq_image": lq_img
        }

train_dataset = ECGDataset(metadata=train_grid, path=path)
val_dataset = ECGDataset(metadata=val_grid, path=path, suffix_map=val_suffix_map)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)

In [ ]:
def unet_document_crop(batch_lq_imgs, model, device):
    cropped_lq_list = []
    masks_up = []
    
    model.to(device)
    model.eval()
    
    with torch.no_grad():
        for lq in batch_lq_imgs:

            # Convert to Numpy
            if isinstance(lq, torch.Tensor):
                lq = lq.cpu().numpy()

            # Downsample
            small_lq = downsample(lq)

            # Convert to Tensor and Normalise for Unet1
            tensor = torch.from_numpy(small_lq).unsqueeze(0)  # (1, H, W)
            tensor = tensor.float() / 255.0                   
            tensor = transforms.Normalize(mean=(UNET1_MEAN / 255.0), std=(UNET1_STD / 255.0))(tensor)
            tensor = tensor.unsqueeze(0).to(device)           # (1, 1, H, W)

            # UNet1 Inference- predict and upsample mask
            logits = model(tensor)
            mask_small = (torch.sigmoid(logits)[0, 0] > 0.5).cpu().numpy().astype(np.uint8)
            mask_up = cv2.resize(mask_small, (2200, 1700), interpolation=cv2.INTER_NEAREST)
            masks_up.append(mask_up)

            # Crop LQ with Mask
            cropped = crop_with_mask(lq, mask_up)
            cropped_lq_list.append(cropped)

    return cropped_lq_list, masks_up


def crop_with_mask(padded_img, mask_up):

    mask_uint8 = (mask_up > 0).astype(np.uint8) * 255

    # Clean mask
    kernel = np.ones((15, 15), np.uint8)
    mask = cv2.morphologyEx(mask_uint8, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return padded_img

    largest = max(contours, key=cv2.contourArea)

    # Best-fit rotated rectangle
    rect = cv2.minAreaRect(largest)
    box = cv2.boxPoints(rect).astype(np.float32)
    pts = order_corners_clockwise(box)

    dst = np.float32([
        [0, 0],
        [2200 - 1, 0],
        [2200 - 1, 1700 - 1],
        [0, 1700 - 1]
    ])

    M = cv2.getPerspectiveTransform(pts, dst)
    warped = cv2.warpPerspective(padded_img, M, (2200, 1700))

    return warped


def to_tensor(img):
    
        # FFT Channel
        f = np.fft.fft2(img)
        fshift = np.fft.fftshift(f)
        fft_mag = np.log(np.abs(fshift)+1)
        fft_mag = (fft_mag - fft_mag.min()) / (fft_mag.max()-fft_mag.min())

        # Image Channel + Concat
        lq_patch_norm = (img/255).astype(np.float32)
        lq_tensor = torch.from_numpy(np.stack([lq_patch_norm, fft_mag], axis=0)).float()
        lq_tensor[0] = (lq_tensor[0]-UNET2_MEAN)/UNET2_STD
    
        return lq_tensor

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

unet2_path = "/kaggle/input/unet2-ecg-mask/latest_unet2_79epoch.pth"
unet = UNet_Res_CBAM()
unet.load_state_dict(torch.load(unet2_path, map_location="cpu"))

unet.to(device)
unet.eval()

In [ ]:
# Retrieve model state from Drive and load
unet2 = "1SqxekqqNbIMwOrPAqRdXdVGXYPIbMSBu"
unet2_pth = f"https://drive.google.com/uc?id={unet2}"
gdown.download(unet2_pth, "unet2.pth", quiet=False)

unet = UNet_Res_CBAM()
unet.load_state_dict(torch.load("unet2.pth", map_location="cpu"))

device = "cuda" if torch.cuda.is_available() else "cpu"
unet.to(device)
unet.eval()

# Retrieve optimizer state from Drive and load
checkpoint = "1DBT4i3epRdqU7_Eh6EcHeXY6DtYG4XfX"
chck_pth = f"https://drive.google.com/uc?id={checkpoint}"
gdown.download(chck_pth, "chck.pth", quiet=False)
checkpoint = torch.load("chck.pth")

optimizer= optim.Adam(unet.parameters(), lr=1e-4)
optimizer.load_state_dict(checkpoint["optimizer"])


In [ ]:
torch.cuda.manual_seed(0)
torch.cuda.empty_cache()
torch.cuda.synchronize()

device = "cuda" if torch.cuda.is_available() else "cpu"

unet = UNet_Res_CBAM().to(device)
optimizer = optim.Adam(unet.parameters(), lr=1e-4)
criterion = DiceFocalLoss()

num_epochs = 40
best_val_loss = float("inf")
train_losses, val_losses = [], []
best_model_wts = copy.deepcopy(unet.state_dict())
best_optim_wts = copy.deepcopy(optimizer.state_dict())

for epoch in range(num_epochs):
    
    # TRAIN 
    unet.train()
    total_train_loss = 0.0

    train_pbar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Training", leave=False)
    for batch in train_pbar:
        lq_imgs = batch["lq_image"] 
        
        id_batch = batch["id"]
        fs_batch = batch["fs"]

        batch_lq_tensors = []
        batch_hq_tensors = []

        # Crop images with UNet1
        cropped_lq_list, masks_up = unet_document_crop(lq_imgs, model=model, device=device)

        # Random ECG patch
        trace_ranges = {0:(510, 894), 
                        1: (797, 1181), 
                        2:(1076, 1460), 
                        3:(1316, 1700)}
        
        trace_ranges = {0:(406, 998), 
                    1: (693, 1285), 
                    2:(972, 1564), 
                    3:(1108, 1700)}

        
        
        for lq_cropped, id_str, fs in zip(cropped_lq_list, id_batch, fs_batch):
            key, y_values = random.choice(list(trace_ranges.items()))

            # Crop LQ patch
            lq_patch = lq_cropped[y_values[0]:y_values[1], 4:-4]
            lq_tensor = to_tensor(lq_patch)  
            batch_lq_tensors.append(lq_tensor)

            # Crop HQ patch 
            df = pd.read_csv(f'{path}/{id_str}/{id_str}.csv')
            signals = df.values
            hq_img = render_ecg(signals, fs, key=key)
            hq_img = hq_img.astype(np.float32)
            hq_img = (hq_img - hq_img.min()) / (hq_img.max() - hq_img.min())
            hq_img = (hq_img > 0.5).astype(np.float32)

            kernel = np.ones((3, 3), np.uint8)   
            hq_img = cv2.dilate(hq_img, kernel, iterations=1)
            hq_patch = hq_img[y_values[0]:y_values[1], 4:-4]
            hq_patch = torch.tensor(hq_patch)
            batch_hq_tensors.append(hq_patch.unsqueeze(0).float())

        # Stack batch and move to device
        batch_lq_tensors = torch.stack(batch_lq_tensors).to(device)
        batch_hq_tensors = torch.stack(batch_hq_tensors).to(device)

        # Forward and backward pass on UNet2
        optimizer.zero_grad()
        hq_pred = unet(batch_lq_tensors)
        loss = criterion(hq_pred, batch_hq_tensors)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    del batch_lq_tensors, batch_hq_tensors, hq_pred, loss
    torch.cuda.empty_cache()

    avg_train_loss = total_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # VALIDATION 
    unet.eval()
    total_val_loss = 0.0
    trace_idx = 0
    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Evaluating", leave=False)
        for batch in val_pbar:
            lq_imgs = batch["lq_image"]
            
            id_batch = batch["id"]
            fs_batch = batch["fs"]

            batch_lq_tensors = []
            batch_hq_tensors = []
    
            # Crop images with UNet1
            cropped_lq_list, masks_up = unet_document_crop(lq_imgs, model=model, device=device)
    
            # Random ECG patch
            trace_ranges = {0:(510, 894), 
                        1: (797, 1181), 
                        2:(1076, 1460), 
                        3:(1316, 1700)}

            trace_ranges = {0:(406, 998), 
                    1: (693, 1285), 
                    2:(972, 1564), 
                    3:(1108, 1700)}

            
            
            for lq_cropped, id_str, fs in zip(cropped_lq_list, id_batch, fs_batch):
                key = trace_idx % len(trace_ranges)
                y_values = trace_ranges[key]
                
                trace_idx += 1
    
                # Crop LQ patch
                lq_patch = lq_cropped[y_values[0]:y_values[1], 4:-4]
                lq_tensor = to_tensor(lq_patch)  
                batch_lq_tensors.append(lq_tensor)
    
                # Crop HQ patch 
                df = pd.read_csv(f'{path}/{id_str}/{id_str}.csv')
                signals = df.values
                hq_img = render_ecg(signals, fs, key=key)
                hq_img = hq_img.astype(np.float32)
                hq_img = (hq_img - hq_img.min()) / (hq_img.max() - hq_img.min())
                hq_img = (hq_img > 0.5).astype(np.float32)

                kernel = np.ones((3, 3), np.uint8)   
                hq_img = cv2.dilate(hq_img, kernel, iterations=1)
                hq_patch = hq_img[y_values[0]:y_values[1], 4:-4]
                hq_patch = torch.tensor(hq_patch)
                batch_hq_tensors.append(hq_patch.unsqueeze(0).float())
                
            # Stack batch and move to device
            batch_lq_tensors = torch.stack(batch_lq_tensors).to(device)
            batch_hq_tensors = torch.stack(batch_hq_tensors).to(device)

            # Forward pass
            hq_pred = unet(batch_lq_tensors)
            val_loss = criterion(hq_pred, batch_hq_tensors)
            total_val_loss += val_loss.item()

    del batch_lq_tensors, batch_hq_tensors, hq_pred
    torch.cuda.empty_cache()

    avg_val_loss = total_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")

    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_wts = copy.deepcopy(unet.state_dict())
        best_optim_wts = copy.deepcopy(optimizer.state_dict())

# Load best weights
unet.load_state_dict(best_model_wts)
optimizer.load_state_dict(best_optim_wts)
unet.eval()

# Optionally, plot losses
plt.figure(figsize=(8,5))
plt.plot(train_losses, label="Train Loss", marker='o')
plt.plot(val_losses, label="Val Loss", marker='s')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
unet.load_state_dict(best_model_wts)
optimizer.load_state_dict(best_optim_wts)

torch.save(unet.state_dict(), "/kaggle/working/latest_unet2_100epoch.pth")
checkpoint = {
    "epoch":epoch,
    "optimizer": optimizer.state_dict()
}
torch.save(checkpoint, "latest_chck_unet2_100epoch.pth")

# ORB Crop + Grid Removal

In [ ]:
def to_img(t):
    t = t.detach().cpu().squeeze().numpy()
    return t

    
def unet_cascade(lq_img):
    unet.eval()
    model.eval()

    if lq_img.dim() == 2: 
        lq_img = lq_img.unsqueeze(0)

    # UNet1
    cropped_lq, mask_up = unet_document_crop(lq_img, model=model, device=device)

    # Rotation
    cropped_lq = [
        cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE) if img.shape[0] > img.shape[1] else img
        for img in cropped_lq
    ]
    
    trace_ranges = [
    (510, 894),
    (797, 1181),
    (1076, 1460),
    (1316, 1700),
    ]

    lq_tensors = []

    for lq_cropped in cropped_lq:
        for y_vals in trace_ranges:
    
            # LQ patch
            lq_patch = lq_cropped[y_vals[0]:y_vals[1], 4:-4]
            lq_tensor = to_tensor(lq_patch)
            lq_tensors.append(lq_tensor)


    preds_list = []
    for i in range(0, len(lq_tensors), 2):
        batch = lq_tensors[i : i + 2]  
        batch = torch.stack(batch).to(device)

        with torch.no_grad():
            logits = unet(batch)                
            probs  = torch.sigmoid(logits)
            preds  = (probs > 0.5).float()

        preds_list.extend(preds.cpu())
        
    img_preds = [to_img(pred) for pred in preds_list]
    
    return img_preds

In [ ]:
def unet_cascade_with_orientation_fix(lq_img, baseline_search=50):

    def check_strip_orientation(strip):

        bin_img = strip * 255 if strip.max() <= 1 else strip

        hist = np.sum(bin_img == 255, axis=1)
        baseline_row = np.argmax(hist)

        h = bin_img.shape[0]

        white_above = np.any(bin_img[0:baseline_row-50, :] > 0)
        white_below = np.any(bin_img[baseline_row+50:, :] > 0)

        return True if white_below or white_above else False

    preds = unet_cascade(lq_img)  
    any_upside_down = any(not check_strip_orientation(strip) for strip in preds)

    if any_upside_down:
      
        if isinstance(lq_img, torch.Tensor):
            img_np = lq_img.squeeze().cpu().numpy()
        else:
            img_np = lq_img

        inverted_np = cv2.rotate(img_np, cv2.ROTATE_180)

        if isinstance(lq_img, torch.Tensor):
            inverted_tensor = torch.from_numpy(inverted_np).to(lq_img.device)
            if inverted_tensor.dim() == 2:
                inverted_tensor = inverted_tensor.unsqueeze(0)
            final_preds = unet_cascade(inverted_tensor)
        else:
            final_preds = unet_cascade(inverted_np)
    else:
        final_preds = preds

    return final_preds


# Post-processing

In [ ]:
def erode(img):
    #kernel = np.ones((3, 3), np.uint8)
    #img = cv2.erode(img, kernel, iterations=1)

    kernel = np.ones((2, 2), np.uint8)
    img = cv2.morphologyEx(img, cv2.MORPH_OPEN, kernel)
    return img


In [ ]:
def extract_ecg_trace(img: np.ndarray):
    binary_img = (img > 0).astype(np.uint8)
    height, width = binary_img.shape

    # Find global baseline (the row with the most pixels)
    #hist = np.sum(binary_img, axis=1)
    #baseline_row = np.argmax(hist)

    # Calculate baseline from MEDIAN
    all_trace_pixels = []
    for col in range(width):
        rows_with_signal = np.where(binary_img[:, col] > 0)[0]
        all_trace_pixels.extend(rows_with_signal)
    
    baseline_row = int(np.median(all_trace_pixels)) if all_trace_pixels else height // 2

    trace_rows = []
    
    cols_with_data = np.where(np.any(binary_img > 0, axis=0))[0]
    
    if len(cols_with_data) == 0:
        return [np.nan]*width, [np.nan]*width, baseline_row

    first_col = cols_with_data[0]
    last_col = cols_with_data[-1]

    for col in range(width):
        rows_with_signal = np.where(binary_img[:, col] > 0)[0]

        if len(rows_with_signal) == 0:
            # If we are inside the active signal range, use baseline
            if first_col < col < last_col:
                selected_row = baseline_row
            else:
                # Outside the signal range (padding), use NaN
                selected_row = np.nan
        else:
            # Standard extraction: find pixel furthest from baseline
            deviations = np.abs(rows_with_signal - baseline_row)
            selected_row = rows_with_signal[np.argmax(deviations)]

        trace_rows.append(selected_row)

    voltages = [
        (baseline_row - y) * 0.5 / 39 if not np.isnan(y) else np.nan 
        for y in trace_rows
    ]

    return trace_rows, voltages, baseline_row

In [ ]:
def resample_to_length(voltage_img, target_len):
    
    voltage_img = np.array(voltage_img)

    # Original index 
    x_old = np.linspace(0, 1, len(voltage_img))

    # Target index 
    x_new = np.linspace(0, 1, target_len)

    # Interpolate
    #f = interp1d(x_old, voltage_img, kind='linear', fill_value='extrapolate')
    f = interp1d(x_old, voltage_img, kind='cubic', fill_value='extrapolate')
    v_resampled = f(x_new)

    return v_resampled


# Full Pipeline

In [ ]:
from scipy.signal import correlate

def aligned_mse(pred, target, fs, max_shift_sec=0.2):

    # Truncate to equal length
    L = min(len(pred), len(target))
    pred = pred[:L]
    target = target[:L]

    # ---- TIME ALIGNMENT ----
    max_shift = int(max_shift_sec * fs)

    # Cross-correlation
    corr = correlate(target, pred, mode="full")
    lags = np.arange(-len(pred) + 1, len(target))

    # Limit lags to ±max_shift
    valid = np.where((lags >= -max_shift) & (lags <= max_shift))[0]
    best_lag = lags[valid][np.argmax(corr[valid])]

    # Apply horizontal shift
    if best_lag > 0:
        pred_shifted = pred[best_lag:]
        target_shifted = target[:len(pred_shifted)]
    elif best_lag < 0:
        target_shifted = target[-best_lag:]
        pred_shifted = pred[:len(target_shifted)]
    else:
        pred_shifted = pred
        target_shifted = target

    # ---- VERTICAL SHIFT CORRECTION ----
    offset = np.mean(target_shifted - pred_shifted)
    pred_corrected = pred_shifted + offset

    # ---- FINAL MSE ----
    mse = np.mean((pred_corrected - target_shifted)**2)

    # Cross correlation
    p_norm = (pred_shifted - np.mean(pred_shifted)) / (np.std(pred_shifted) + 1e-8)
    t_norm = (target_shifted - np.mean(target_shifted)) / (np.std(target_shifted) + 1e-8)
    
    sync_agnostic_corr = np.mean(p_norm * t_norm)

    return mse, best_lag/fs, offset, sync_agnostic_corr


In [ ]:
##### FIGURE 1 #####
device = "cuda" if torch.cuda.is_available() else "cpu"

# Target sample IDs for visualization
target_ids = ('987816877', '3117857513', '1561472702', '1962423499')

# Collect samples with target IDs
selected_samples = []
model.eval()

for batch in val_loader:
    batch_ids = batch['id']
    for idx, sample_id in enumerate(batch_ids):
        if sample_id in target_ids:
            selected_samples.append({
                'id': sample_id,
                'img': batch['lq_image'][idx:idx+1],
            })
            if len(selected_samples) == len(target_ids):
                break
    if len(selected_samples) == len(target_ids):
        break

# Generate predictions and create visualization
N = len(selected_samples)

selected_samples = torch.cat([s['img'] for s in selected_samples], dim=0)
with torch.no_grad():
    fig, axes = plt.subplots(N, 3, figsize=(10, 4 * N), dpi=300)
    if N == 1:
        axes = axes.reshape(1, -1)
    # Get prediction
    croppeds, masks = unet_document_crop(selected_samples, model, device)
    masks = [mask*255 for mask in masks]
    for i, (img, overlay_pred, cropped) in enumerate(zip(selected_samples, masks, croppeds)):
        img_np = img

        # Plot
        axes[i, 0].imshow(img_np, cmap='gray')
        axes[i, 0].set_title("Input Image", fontsize=12)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(img_np, cmap='gray')
        axes[i, 1].imshow(overlay_pred, cmap='Blues',alpha=0.7)
        axes[i, 1].set_title("Predicted Mask", fontsize=12)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(cropped, cmap='gray')
        axes[i, 2].set_title("Cropped Document", fontsize=12)
        axes[i, 2].axis('off')
        
plt.tight_layout()
plt.savefig('figure1.png', dpi=300, bbox_inches='tight', facecolor='white', pad_inches=0.1)
print("Saved visualization to 'predictions_visualization.png'")
plt.show()

In [ ]:
### FIGURE 2 ###
met = pd.read_csv('/kaggle/input/physionet-ecg-image-digitization/train.csv')
selected_samples = []
target_ids = {'987816877', '3117857513', '1561472702', '1962423499'}

for batch in val_loader:
    batch_ids = batch['id']
    for idx, sample_id in enumerate(batch_ids):
        if sample_id in target_ids:
            selected_samples.append({
                'id': sample_id,
                'img': batch['lq_image'][idx:idx+1],
            })
            if len(selected_samples) == len(target_ids):
                break
    if len(selected_samples) == len(target_ids):
        break

# A4 paper width in inches (8.27 inches)
paper_width = 8.27

for sample in selected_samples:
    sample_id = sample['id']
    lq_img = torch.tensor(sample['img'])

    sample_fs = met[met['id']==int(sample_id)]['fs'].values[0]
    sample_len = met[met['id']==int(sample_id)]['sig_len'].values[0]
    
    sample_len_short = int(sample_len/4)
    times_orig = np.arange(sample_len) / sample_fs
    times_orig_short = np.arange(sample_len_short) / sample_fs
    
    # Load ground truth
    sample_csv = pd.read_csv(
        f"/kaggle/input/physionet-ecg-image-digitization/train/{sample_id}/{sample_id}.csv"
    )
    
    # UNet inference
    img_preds = unet_cascade_with_orientation_fix(lq_img)
    imgs_eroded = [erode(pred) for pred in img_preds]
    components = [(eroded) for eroded in imgs_eroded]
    
    labels = {
        0: ['I', 'aVR', 'V1', 'V4'],
        1: ['II', 'aVL', 'V2', 'V5'],
        2: ['III', 'aVF', 'V3', 'V6'],
        3: ['II']      
    }
    
    # Create comprehensive figure
    fig = plt.figure(figsize=(paper_width, 14), dpi=300)
    gs = fig.add_gridspec(10, 4, hspace=0.4, wspace=0.2, 
                          height_ratios=[0, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 1.2])
    
    # 1. Input Image (full width)
    ax_input = fig.add_subplot(gs[0:5, :3])
    input_img = lq_img.squeeze().cpu().numpy()
    ax_input.imshow(input_img, cmap='gray', alpha=0)
    ax_input.axis('off')
    
    # 2. Process each strip and its leads
    all_lead_data = []

    for i, component in enumerate(components):
        strip = component[:, 4:-4]
        
        if i != 3:  # First 3 strips (4 leads each)
            # Show strip
            ax_strip = fig.add_subplot(gs[i+1, :])
            ax_strip.imshow(strip, cmap='gray')
            ax_strip.set_title(f'Strip {i+1}', fontsize=9)
            ax_strip.axis('off')
            
            # Extract trace
            trace_pixels, voltages, baseline_row = extract_ecg_trace(strip)
            voltages = np.array(voltages)
            v_notnan = voltages[~np.isnan(voltages)]
            v = resample_to_length(v_notnan, sample_len)
            
            start_idx = 0
            
            for j in range(4):
                orig = sample_csv[labels[i][j]]
                seg = v[start_idx:start_idx + sample_len_short]              

                if len(orig) > (sample_len_short):
                    orig = orig[start_idx:start_idx + sample_len_short]
                    
                start_idx += sample_len_short
                mse, _, _, _ = aligned_mse(seg, orig, sample_fs)
                all_lead_data.append({
                    'strip': i,
                    'lead': labels[i][j],
                    'seg': seg,
                    'orig': orig,
                    'times': np.arange(len(seg))/sample_fs,
                    'mse': mse
                })
        
        else:  # Long strip (1 lead)
            # Show strip (span 2 columns)
            ax_strip = fig.add_subplot(gs[4, :])
            ax_strip.imshow(strip, cmap='gray')
            ax_strip.set_title('Strip 4', fontsize=9)
            ax_strip.axis('off')
            
            # Extract trace
            trace_pixels, voltages, baseline_row = extract_ecg_trace(strip)
            voltages = np.array(voltages)
            v_notnan = voltages[~np.isnan(voltages)]
            v = resample_to_length(v_notnan, sample_len)
            
            orig = sample_csv[labels[i][0]]
            mse, _, _, _ = aligned_mse(v, orig, sample_fs)
            all_lead_data.append({
                'strip': i,
                'lead': labels[i][0],
                'seg': v,
                'orig': orig,
                'times': times_orig,
                'mse': mse
            })
        
    # 3. Plot leads (4x3 grid for 12 leads + 1 long)
    for idx, lead_data in enumerate(all_lead_data):
        if idx < 12:  # First 12 leads in 3 rows
            row = 5 + (idx // 4)
            col = idx % 4
        else:  # 13th lead (long Lead II) spans full width
            row = 8
            col = slice(None)
        
        ax = fig.add_subplot(gs[row, col])
        ax.plot(lead_data['times'], lead_data['seg'], 'r-', linewidth=0.8, label='Predicted', alpha=0.8)
        ax.plot(lead_data['times'], lead_data['orig'], 'g-', linewidth=0.8, label='Ground Truth', alpha=0.8)
        ax.set_title(f"{lead_data['lead']} (MSE: {lead_data['mse']:.4f})", fontsize=8)
        ax.grid(True, alpha=0.3, linewidth=0.5)
        ax.tick_params(labelsize=7)
        
        if idx == 0:
            ax.legend(fontsize=7, loc='upper right')
        
        if row == 4 or row == 5:  # Bottom row
            ax.set_xlabel('Time (s)', fontsize=7)
        if col == 0 or (isinstance(col, slice)):
            ax.set_ylabel('Voltage (mV)', fontsize=7)
    
    
    # Save figure
    plt.savefig(f'ecg_analysis_{sample_id}.png', dpi=300, bbox_inches='tight', 
                facecolor='white', pad_inches=0.1)
    print(f"Saved: ecg_analysis_{sample_id}.png")
    plt.show()
    plt.close()

In [ ]:
### METRICS ###
met = pd.read_csv('/kaggle/input/physionet-ecg-image-digitization/train.csv')

from collections import defaultdict
rmse = defaultdict(list)
corr = defaultdict(list)

for batch in val_loader:
    batch_ids = batch['id']
    batch_imgs = batch['lq_image']

    for sample_id, img in zip(batch_ids, batch_imgs):

        #lq_img = torch.tensor(img)
        lq_img = img
        sample_fs = met[met['id']==int(sample_id)]['fs'].values[0]
        sample_len = met[met['id']==int(sample_id)]['sig_len'].values[0]
    
        sample_len_short = int(sample_len/4)
        times_orig = np.arange(sample_len) / sample_fs
        times_orig_short = np.arange(sample_len_short) / sample_fs
    
        # Load ground truth
        sample_csv = pd.read_csv(
            f"/kaggle/input/physionet-ecg-image-digitization/train/{sample_id}/{sample_id}.csv"
        )
        
        # UNet inference
        img_preds = unet_cascade_with_orientation_fix(lq_img)
        components = [erode(pred) for pred in img_preds]
        
        labels = {
            0: ['I', 'aVR', 'V1', 'V4'],
            1: ['II', 'aVL', 'V2', 'V5'],
            2: ['III', 'aVF', 'V3', 'V6'],
            3: ['II']      
        }
    
        for i, component in enumerate(components):
            strip = component[:, 4:-4]
            
            if i != 3:  # First 3 strips (4 leads each)
                
                # Extract trace
                trace_pixels, voltages, baseline_row = extract_ecg_trace(strip)
                voltages = np.array(voltages)
                v_notnan = voltages[~np.isnan(voltages)]
                v = resample_to_length(v_notnan, sample_len)

                start_idx = 0
                
                for j in range(4):

                    lead = labels[i][j]

                    orig = sample_csv[labels[i][j]]
                        
                    seg = v[start_idx:start_idx + sample_len_short]              
                    
                    if len(orig) > (sample_len_short):
                        orig = orig[start_idx:start_idx + sample_len_short]

                    if np.isnan(orig).any():
                        orig = orig[~np.isnan(orig)]
                    start_idx += sample_len_short

                    mse, _, _, sync_agnostic_corr = aligned_mse(seg, orig, sample_fs)
                    rmse[lead].append(np.sqrt(mse))
                    corr[lead].append(sync_agnostic_corr)
            
            else:  
                # Extract trace
                trace_pixels, voltages, baseline_row = extract_ecg_trace(strip)
                voltages = np.array(voltages)
                v_notnan = voltages[~np.isnan(voltages)]
                if len(v_notnan)==0:
                    plt.imshow(strip)
                    plt.show
                    continue
                v = resample_to_length(v_notnan, sample_len)
                
                orig = sample_csv[labels[i][0]]
                if np.isnan(orig).any():
                    orig = orig[~np.isnan(orig)]

                mse, _, _, sync_agnostic_corr = aligned_mse(v, orig, sample_fs)
                lead = labels[i][0]

                rmse[lead].append(np.sqrt(mse))
                corr[lead].append(sync_agnostic_corr)

total_mse = []
for lead, metrics in rmse.items():
    print('RMSE metrics')
    total_mse.append(np.mean(metrics))
    print(lead, np.mean(metrics))
print('Total mean', np.mean(total_mse))

total_corr = []
for lead, metrics in corr.items():
    print('Cross-correlation metrics')
    total_corr.append(np.mean(metrics))
    print(lead, np.mean(metrics))
print('Total mean', np.mean(total_corr))
    

# Submission parquet

In [ ]:
def show_trace_grid(traces, sample_fs, lead_names):

    layout = [4, 3, 4, 1]
    max_cols = 4

    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(4, max_cols, wspace=0.25, hspace=0.5)

    idx = 0

    for r, count in enumerate(layout):
        
        if r == 3:
            ax = fig.add_subplot(gs[r, :])
            seg = traces[idx]
            t = np.arange(len(seg)) / sample_fs

            ax.plot(t, seg)
            ax.set_title(lead_names[idx])
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("mV")

            idx += 1
            continue

        for c in range(max_cols):
            if r == 1 and c == 0:
                continue

            if idx >= len(traces):
                break

            if c >= count + (1 if r == 1 else 0):
                break

            ax = fig.add_subplot(gs[r, c])

            seg = traces[idx]
            t = np.arange(len(seg)) / sample_fs

            ax.plot(t, seg)
            ax.set_title(lead_names[idx])
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("mV")

            idx += 1

    plt.show()


In [ ]:
path_test = '/kaggle/input/physionet-ecg-image-digitization/test'
metadata_df_test = pd.read_csv('/kaggle/input/physionet-ecg-image-digitization/test.csv')

def preprocess(img_path):
    filename = os.path.basename(img_path)
    id_value = int(filename[:-4])
    meta = metadata_df_test[metadata_df_test['id'] == id_value]
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = resize_with_padding(img)

    return img, id_value, meta

submission_rows = []

labels = {
    0: ['I', 'aVR', 'V1', 'V4'],
    1: ['II', 'aVL', 'V2', 'V5'],
    2: ['III', 'aVF', 'V3', 'V6'],
    3: ['II']
}

for item in os.listdir(path_test):

    img_path = os.path.join(path_test, item)
    img, id_value, meta = preprocess(img_path)

    sample_len = np.max(meta['number_of_rows'])
    sample_fs = meta['fs'].unique()[0]

    tens = torch.tensor(img)
    img_preds = unet_cascade_with_orientation_fix(tens)

    components = [erode(pred) for pred in img_preds]
    
    strips_to_show = []
    traces_to_show = []
    leadnames_to_show = []

    for i, component in enumerate(components):

        strip = component[:, 4:-4]
        strips_to_show.append(strip)

        trace_pixels, voltages, baseline_row = extract_ecg_trace(strip)
        voltages = np.array(voltages)
        v = resample_to_length(voltages[~np.isnan(voltages)], sample_len)

        if i != 3:
            start_idx = 0
            for j in range(4):
                lead = labels[i][j]

                if lead == 'II':
                    continue

                number_of_rows = int(meta[meta['lead'] == lead]['number_of_rows'].values.item())
                seg = v[start_idx:start_idx + number_of_rows]
                start_idx += number_of_rows

                traces_to_show.append(seg)
                leadnames_to_show.append(lead)

                for row_id, val in enumerate(seg):
                    submission_rows.append({
                        'id': f"{id_value}_{row_id}_{lead}",
                        'value': val
                    })
        else:
            lead = 'II'
            traces_to_show.append(v)
            leadnames_to_show.append(lead)

            for row_id, val in enumerate(v):
                submission_rows.append({
                    'id': f"{id_value}_{row_id}_{lead}",
                    'value': val
                })

    # Input image plot
    plt.figure(figsize=(16, 12))
    plt.imshow(img, cmap='gray')
    plt.title("Original Image")
    plt.axis("off")
    plt.show()

    # Prediction plots
    show_trace_grid(traces_to_show, sample_fs, leadnames_to_show)

submission_df = pd.DataFrame(submission_rows)
submission_df.to_parquet('submission.parquet', index=False)

In [ ]:
submission_df.head()

# Extra Figures

In [ ]:
# FFT 

id_str = 10140238
hq_image = f"{path}/{id_str}/{id_str}-0001.png"
hq_image = cv2.imread(hq_image)
img = cv2.cvtColor(hq_image, cv2.COLOR_BGR2GRAY)
h, w = img.shape

def compute_fft(img):
    I = img.astype(float)
    I -= I.mean()
    
    S = np.fft.fftshift(np.fft.fft2(I))
    Slog = np.log1p(np.abs(S))
    return S, Slog

S, Slog = compute_fft(img)

Slog_norm = (Slog - Slog.mean())/ Slog.std()

thr_grad = np.percentile(Slog_norm, 99.5) 
grad_mag = Slog_norm > thr_grad
mask_inv = (~grad_mag)

cx, cy = w // 2, h // 2

U, V = np.meshgrid(np.arange(w) - cx, np.arange(h) - cy)
radius = np.sqrt((U/(w*0.8))**2 + (V/(h*0.8))**2)  
lp = np.exp(-radius**2)   

combined_mask = mask_inv * lp
mask = gaussian_filter(combined_mask, sigma=0.05)

fshift_filtered = S * mask
img_back = np.fft.ifft2(np.fft.ifftshift(fshift_filtered))
img_back = np.abs(img_back)

img_filtered = cv2.normalize(img_back, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

_, binary = cv2.threshold(img_filtered, 35, 255, cv2.THRESH_BINARY)

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2,2))
opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)
closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel, iterations=1)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.imshow(Slog, cmap='hot')
plt.title('Original FFT (log scale)')
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(closed, cmap='gray')
plt.title('Reconstructed (Grid Filtered)')
plt.axis('off')

plt.tight_layout()
plt.savefig(f'fft.png', dpi=300, bbox_inches='tight', 
                facecolor='white', pad_inches=0.1)
plt.show()


In [ ]:
# ORB FIGURE

id_str = 10140238
hq_image = cv2.imread(f"{path}/{id_str}/{id_str}-0001.png")
lq_image = cv2.imread(f"{path}/{id_str}/{id_str}-0006.png")

align = ORB()
kp1, des1 = align.keypoint_descriptors(hq_image)
kp2, des2 = align.keypoint_descriptors(lq_image)
matches_knn = align.bf.knnMatch(des1, des2, k=2)
good_matches = [m for m, n in matches_knn if m.distance < 0.75 * n.distance]
good_matches = sorted(good_matches, key=lambda x: x.distance)
_, H = align.call_kp(kp1, des1, lq_image)

# Scale the HQ image to match LQ image size
scale_factor = h2 / h1  
hq_scaled = cv2.resize(hq_image, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_LINEAR)

h1_scaled, w1_scaled = hq_scaled.shape[:2]
h2, w2 = lq_image.shape[:2]

spacing = 100
canvas_height = max(h1_scaled, h2)
canvas_width = w1_scaled + w2 + spacing
canvas = np.ones((canvas_height, canvas_width, 3), dtype=np.uint8) * 255

canvas[0:h1_scaled, 0:w1_scaled] = hq_scaled
canvas[0:h2, w1_scaled+spacing:w1_scaled+spacing+w2] = lq_image

# Convert to RGB
canvas_rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)

# Create figure
plt.figure(figsize=(22, 8))
plt.subplot(2, 1, 1)

plt.imshow(canvas_rgb)

for match in good_matches[:30]:
    pt1 = kp1[match.queryIdx].pt
    pt1_scaled = (int(pt1[0] * scale_factor), int(pt1[1] * scale_factor))
    
    pt2 = kp2[match.trainIdx].pt
    pt2_shifted = (int(pt2[0]) + w1_scaled + spacing, int(pt2[1]))
    
    plt.plot([pt1_scaled[0], pt2_shifted[0]], [pt1_scaled[1], pt2_shifted[1]], 
             'g-', linewidth=0.5, alpha=0.6)

plt.axis('off')


plt.subplot(2, 1, 2)
plt.imshow(canvas_rgb)
hq_corners = [
    [0, 0],                          
    [w1_scaled, 0], 
    [w1_scaled, h1_scaled], 
    [0, h1_scaled]               
              
]

lq_corners = [
    [5310, 830],
    [8200, 790],                          
    [8200, 2960]  ,
     [5360, 2990],
    [5310, 830],
]

for corner in hq_corners:
    plt.scatter(corner[0], corner[1], c='yellow', s=50, edgecolors='black', zorder=5)

# Draw yellow corner dots on LQ image
for corner in lq_corners:
    plt.scatter(corner[0], corner[1], c='yellow', s=50, edgecolors='black', zorder=5)

corne = np.array(lq_corners)
plt.plot(corne[:, 0], corne[:, 1], 'c--', linewidth=3, label='Dashed Mask Boundary')

for i in range(4):
    x_hq, y_hq = hq_corners[i]
    x_lq, y_lq = lq_corners[i]
    
    # We add w_hq to the x_lq because it's the second image in the side-by-side plot
    plt.plot([x_hq, x_lq], [y_hq, y_lq], 
             
             linestyle='--', 
             linewidth=2, 
             alpha=0.7)

plt.axis('off')

plt.tight_layout()
plt.axis('off')
plt.show()
plt.savefig(
    'ORB_mask.png', 
    dpi=300,                
    bbox_inches='tight',    
    pad_inches=0.1,         
    transparent=False,      
    facecolor='white'       
)